In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/ieee-fraud-detection/sample_submission.csv
/kaggle/input/competitions/ieee-fraud-detection/test_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/test_transaction.csv
/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv


In [2]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

import xgboost as xgb

In [3]:
train_transaction = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv')
train_identity = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv')

train = train_transaction.merge(train_identity, on='TransactionID', how='left')

print(train.shape)

(590540, 434)


In [4]:
cols_to_drop = train.columns[train.isnull().mean() > 0.9]
train = train.drop(columns=cols_to_drop)

for col in train.columns:
    if train[col].dtype == 'object':
        train[col] = train[col].fillna('missing')
    else:
        train[col] = train[col].fillna(-999)

In [5]:
new_features = pd.DataFrame(index=train.index)

new_features['TransactionAmt_log'] = np.log1p(train['TransactionAmt'])
new_features['TransactionDT_hours'] = train['TransactionDT'] / 3600
new_features['TransactionDT_days'] = train['TransactionDT'] / (3600 * 24)

new_features['P_emaildomain_prefix'] = train['P_emaildomain'].astype(str).str.split('.').str[0]

for col in ['card1', 'card2', 'card3', 'card5']:
    freq = train[col].value_counts()
    new_features[col + '_freq'] = train[col].map(freq)

train = pd.concat([train, new_features], axis=1)
train = train.copy()

In [6]:
from sklearn.preprocessing import LabelEncoder

cat_cols = train.select_dtypes(include='object').columns

for col in cat_cols:
    le = LabelEncoder()
    train[col] = le.fit_transform(train[col].astype(str))

In [7]:
X = train.drop(columns=['isFraud', 'TransactionID'])
y = train['isFraud']

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [8]:
model = xgb.XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric='auc',
    use_label_encoder=False,
    tree_method='hist'  
)

model.fit(X_train, y_train)

/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [12:55:02] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='auc', feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.1, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=6,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=200,
              n_jobs=None, num_parallel_tree=None, ...)

In [9]:
y_pred_train = model.predict_proba(X_train)[:, 1]
y_pred_val = model.predict_proba(X_val)[:, 1]

roc_train = roc_auc_score(y_train, y_pred_train)
roc_val = roc_auc_score(y_val, y_pred_val)

print("Train ROC:", roc_train)
print("Val ROC:", roc_val)

Train ROC: 0.9544648441128475
Val ROC: 0.9423581887485089


In [10]:
!pip install mlflow dagshub

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 67.0 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 73.9 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 47.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 273.1/273.1 kB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.4/208.4 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.2/132.2 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [11]:
import dagshub
dagshub.init(repo_owner='slosa23', repo_name='ML-Assignment2', mlflow=True)

❗❗❗ AUTHORIZATION REQUIRED ❗❗❗



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=7f20df9c-961b-416a-954f-9477be3bd38d&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=b9b85e370e00f1e36a4d96d139ff9c35e488a1fcdaa3a60f2de1f12c0257b59b




Output()

Accessing as slosa23

Initialized MLflow to track repo "slosa23/ML-Assignment2"

Repository slosa23/ML-Assignment2 initialized!

In [12]:
import mlflow

mlflow.set_experiment("XGBoost_Training")

with mlflow.start_run(run_name="XGB_Baseline"):

    mlflow.log_param("model", "XGBoost")
    mlflow.log_param("n_estimators", 200)
    mlflow.log_param("max_depth", 6)
    mlflow.log_param("learning_rate", 0.1)
    mlflow.log_param("subsample", 0.8)
    mlflow.log_param("colsample_bytree", 0.8)

    mlflow.log_metric("roc_auc_train", roc_train)
    mlflow.log_metric("roc_auc_val", roc_val)

    mlflow.xgboost.log_model(model, "model")

    print("XGBoost baseline logged")

2026/05/03 13:04:03 INFO mlflow.tracking.fluent: Experiment with name 'XGBoost_Training' does not exist. Creating a new experiment.
2026/05/03 13:04:06 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


XGBoost baseline logged
🏃 View run XGB_Baseline at: https://dagshub.com/slosa23/ML-Assignment2.mlflow/#/experiments/1/runs/77ab91ef03d8492590b5743a6d013c12
🧪 View experiment at: https://dagshub.com/slosa23/ML-Assignment2.mlflow/#/experiments/1


In [13]:
depths = [4, 6, 8]

for depth in depths:
    
    with mlflow.start_run(run_name=f"XGB_depth_{depth}"):

        model = xgb.XGBClassifier(
            n_estimators=200,
            max_depth=depth,
            learning_rate=0.1,
            subsample=0.8,
            colsample_bytree=0.8,
            eval_metric='auc',
            tree_method='hist'
        )

        model.fit(X_train, y_train)

        y_pred = model.predict_proba(X_val)[:, 1]
        roc = roc_auc_score(y_val, y_pred)

        mlflow.log_param("max_depth", depth)
        mlflow.log_metric("roc_auc_val", roc)

        print(f"depth={depth} | ROC={roc:.4f}")

depth=4 | ROC=0.9179
🏃 View run XGB_depth_4 at: https://dagshub.com/slosa23/ML-Assignment2.mlflow/#/experiments/1/runs/47861de783d04ef18032514c5e59b937
🧪 View experiment at: https://dagshub.com/slosa23/ML-Assignment2.mlflow/#/experiments/1
depth=6 | ROC=0.9424
🏃 View run XGB_depth_6 at: https://dagshub.com/slosa23/ML-Assignment2.mlflow/#/experiments/1/runs/f560aeed654647feb2e28532ffbd728d
🧪 View experiment at: https://dagshub.com/slosa23/ML-Assignment2.mlflow/#/experiments/1
depth=8 | ROC=0.9593
🏃 View run XGB_depth_8 at: https://dagshub.com/slosa23/ML-Assignment2.mlflow/#/experiments/1/runs/8586d7a8644747f9afe754dc01341b4e
🧪 View experiment at: https://dagshub.com/slosa23/ML-Assignment2.mlflow/#/experiments/1


In [14]:
lrs = [0.05, 0.1, 0.2]

for lr in lrs:
    
    with mlflow.start_run(run_name=f"XGB_lr_{lr}"):

        model = xgb.XGBClassifier(
            n_estimators=200,
            max_depth=6,
            learning_rate=lr,
            subsample=0.8,
            colsample_bytree=0.8,
            eval_metric='auc',
            tree_method='hist'
        )

        model.fit(X_train, y_train)

        y_pred = model.predict_proba(X_val)[:, 1]
        roc = roc_auc_score(y_val, y_pred)

        mlflow.log_param("learning_rate", lr)
        mlflow.log_metric("roc_auc_val", roc)

        print(f"lr={lr} | ROC={roc:.4f}")

lr=0.05 | ROC=0.9277
🏃 View run XGB_lr_0.05 at: https://dagshub.com/slosa23/ML-Assignment2.mlflow/#/experiments/1/runs/a733601c9f604ae6afc1987328f10356
🧪 View experiment at: https://dagshub.com/slosa23/ML-Assignment2.mlflow/#/experiments/1
lr=0.1 | ROC=0.9424
🏃 View run XGB_lr_0.1 at: https://dagshub.com/slosa23/ML-Assignment2.mlflow/#/experiments/1/runs/d385b3dc5bb0477cbecda08805acb037
🧪 View experiment at: https://dagshub.com/slosa23/ML-Assignment2.mlflow/#/experiments/1
lr=0.2 | ROC=0.9552
🏃 View run XGB_lr_0.2 at: https://dagshub.com/slosa23/ML-Assignment2.mlflow/#/experiments/1/runs/884bf13ce60747779246173c3f18edca
🧪 View experiment at: https://dagshub.com/slosa23/ML-Assignment2.mlflow/#/experiments/1


In [15]:
estimators = [100, 200, 400]

for n in estimators:
    
    with mlflow.start_run(run_name=f"XGB_estimators_{n}"):

        model = xgb.XGBClassifier(
            n_estimators=n,
            max_depth=6,
            learning_rate=0.1,
            subsample=0.8,
            colsample_bytree=0.8,
            eval_metric='auc',
            tree_method='hist'
        )

        model.fit(X_train, y_train)

        y_pred = model.predict_proba(X_val)[:, 1]
        roc = roc_auc_score(y_val, y_pred)

        mlflow.log_param("n_estimators", n)
        mlflow.log_metric("roc_auc_val", roc)

        print(f"n_estimators={n} | ROC={roc:.4f}")

n_estimators=100 | ROC=0.9274
🏃 View run XGB_estimators_100 at: https://dagshub.com/slosa23/ML-Assignment2.mlflow/#/experiments/1/runs/916ddafef1a44d8c90f01c2f246cf1c9
🧪 View experiment at: https://dagshub.com/slosa23/ML-Assignment2.mlflow/#/experiments/1
n_estimators=200 | ROC=0.9424
🏃 View run XGB_estimators_200 at: https://dagshub.com/slosa23/ML-Assignment2.mlflow/#/experiments/1/runs/901c9bea20ed4ebb91f821deaa40a3d1
🧪 View experiment at: https://dagshub.com/slosa23/ML-Assignment2.mlflow/#/experiments/1
n_estimators=400 | ROC=0.9557
🏃 View run XGB_estimators_400 at: https://dagshub.com/slosa23/ML-Assignment2.mlflow/#/experiments/1/runs/18f5e056b872406186e6e0843ab6592c
🧪 View experiment at: https://dagshub.com/slosa23/ML-Assignment2.mlflow/#/experiments/1


In [16]:
best_model = xgb.XGBClassifier(
    n_estimators=400,
    max_depth=6,
    learning_rate=0.2,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric='auc',
    tree_method='hist'
)

best_model.fit(X_train, y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='auc', feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.2, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=6,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=400,
              n_jobs=None, num_parallel_tree=None, ...)

In [17]:
y_pred_train = best_model.predict_proba(X_train)[:, 1]
y_pred_val = best_model.predict_proba(X_val)[:, 1]

roc_train = roc_auc_score(y_train, y_pred_train)
roc_val = roc_auc_score(y_val, y_pred_val)

print("Train ROC:", roc_train)
print("Val ROC:", roc_val)

Train ROC: 0.9888436496968889
Val ROC: 0.9658904225520699


In [18]:
from sklearn.pipeline import Pipeline

pipeline = Pipeline([
    ("model", best_model)
])

In [20]:
with mlflow.start_run(run_name="XGB_BEST"):

    mlflow.log_param("model", "XGBoost")
    mlflow.log_param("n_estimators", 400)
    mlflow.log_param("max_depth", 6)
    mlflow.log_param("learning_rate", 0.2)

    mlflow.log_metric("roc_auc_train", roc_train)
    mlflow.log_metric("roc_auc_val", roc_val)

    mlflow.sklearn.log_model(pipeline, name="model")

    print("Best XGBoost model logged.")

2026/05/03 14:04:11 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Best XGBoost model logged.
🏃 View run XGB_BEST at: https://dagshub.com/slosa23/ML-Assignment2.mlflow/#/experiments/1/runs/6cf50cf2c0b6408cba85a20754c5bf06
🧪 View experiment at: https://dagshub.com/slosa23/ML-Assignment2.mlflow/#/experiments/1
